In [1]:
import sqlite3
from pathlib import Path
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

DB_YOLU = Path("../data/processed/steam.db")
print("Dosya var mı:", DB_YOLU.exists(), "| Boyut:", round(DB_YOLU.stat().st_size / 1e9, 2), "GB")

baglanti = sqlite3.connect(DB_YOLU)

tablolar = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type = 'table' AND name NOT LIKE 'sqlite_%' ORDER BY name",
    baglanti,
)["name"].tolist()

for tablo in tablolar:
    satir = baglanti.execute(f"SELECT COUNT(*) FROM {tablo}").fetchone()[0]
    sutunlar = pd.read_sql(f"PRAGMA table_info({tablo})", baglanti)[["name", "type", "notnull", "pk"]]
    print(f"\n=== {tablo}: {satir:,} satır, {len(sutunlar)} sütun ===")
    print(sutunlar.to_string(index=False))

Dosya var mı: True | Boyut: 14.84 GB

=== oyun_dil: 341,659 satır, 3 sütun ===
   name    type  notnull  pk
 app_id INTEGER        1   1
    dil    TEXT        1   2
ses_var INTEGER        0   0

=== oyun_etiket: 730,709 satır, 4 sütun ===
  name    type  notnull  pk
app_id INTEGER        1   1
etiket    TEXT        1   2
    oy INTEGER        0   0
  sira INTEGER        0   0

=== oyun_kategori: 235,168 satır, 2 sütun ===
    name    type  notnull  pk
  app_id INTEGER        1   1
kategori    TEXT        1   2

=== oyun_tur: 174,563 satır, 2 sütun ===
  name    type  notnull  pk
app_id INTEGER        1   1
   tur    TEXT        1   2

=== oyunlar: 64,948 satır, 36 sütun ===
                    name    type  notnull  pk
                  app_id INTEGER        0   1
                      ad    TEXT        0   0
      steam_cikis_tarihi    TEXT        0   0
                   fiyat    REAL        0   0
           indirim_yuzde INTEGER        0   0
            normal_fiyat    REAL        

### steam.db şeması (LLM'in göreceği yapı)
- 6 tablo, satırlar 03 ile aynı: oyunlar 64.948 (36 sütun), oyun_tur 174.563, oyun_etiket 730.709, oyun_kategori 235.168, oyun_dil 341.659, yorumlar 31.690.738 (15 sütun). Boyut 14,84 GB.
- Yan tablolarda birincil anahtar (app_id, değer) birlikte. oyunlar.app_id ve yorumlar.review_id'nin notnull = 0 görünmesi SQLite özelliği, veride boş anahtar yok.
- **Şema tek başına yetmez, araç açıklamasına yazılacak tuzaklar:**
  - Oran için sadece `oyunlar.olumlu_oran` (olumlu_json/csv ve yorumlar.olumlu değil). Değeri 0–1, yüzde değil.
  - `oyun_dil.dil` = games.json, geliştirici beyanı, "Turkish" gibi. `yorumlar.dil` = fastText, "tr" gibi kod.
  - Popülerlik için `oy_sayisi`, `yorum_sayisi` değil (CSV satırı).
  - Tarihler 'YYYY-AA-GG' metin, aralıkla filtrelenir. Bayraklar 1/0.
  - Uzun metin sütunları (kisa_aciklama, oyun_hakkinda, metin): SELECT * bağlamı doldurur.

In [2]:
import time

def sql_calistir(sorgu):
    baslangic = time.perf_counter()
    try:
        df = pd.read_sql(sorgu, baglanti)
    except Exception as hata:
        return f"HATA: {hata}"
    sure = time.perf_counter() - baslangic
    return f"{len(df)} satır, {sure:.2f} sn\n{df.to_string(index=False)}"

testler = [
    "SELECT ad, olumlu_oran, oy_sayisi FROM oyunlar WHERE ad = 'Dying Light'",
    "SELECT COUNT(*) AS turkce_yorum FROM yorumlar WHERE dil = 'tr'",
    "SELECT ad, puan FROM oyunlar LIMIT 3",
]

for sorgu in testler:
    print("SORGU:", sorgu)
    print(sql_calistir(sorgu))
    print("-" * 60)

SORGU: SELECT ad, olumlu_oran, oy_sayisi FROM oyunlar WHERE ad = 'Dying Light'
1 satır, 2.82 sn
         ad  olumlu_oran  oy_sayisi
Dying Light        0.952     435283
------------------------------------------------------------
SORGU: SELECT COUNT(*) AS turkce_yorum FROM yorumlar WHERE dil = 'tr'
1 satır, 0.02 sn
 turkce_yorum
       619054
------------------------------------------------------------
SORGU: SELECT ad, puan FROM oyunlar LIMIT 3
HATA: Execution failed on sql 'SELECT ad, puan FROM oyunlar LIMIT 3': no such column: puan
------------------------------------------------------------


### Ham sorgu fonksiyonu: sql_calistir(sorgu)
- SQL metnini alır, sonucu **metin** olarak döndürür (LLM DataFrame göremez): ilk satır "N satır, X sn", altında tablo.
- Hata olursa çökmez, `HATA: ...` metni döner (ör. "no such column: puan") → LLM okuyup sorgusunu düzeltebilir.
- Sonuçlar:
  - Dying Light: olumlu_oran 0.952 (%95,2), 435.283 oy. **2,82 sn**: `ad` sütununda indeks yok (64.948 satırın hepsi taranıyor), satırlarda uzun açıklamalar var, üstelik ilk sorgu (veri diskten okundu).
  - Türkçe yorum: 619.054 (03 ile aynı), 0,02 sn, yorumlar(dil) indeksi sayesinde.
- Not: olumlu_oran 0–1 arası; kullanıcıya yüzde olarak söylenmeli. Ad ile arama hızı risk ölçümünde ayrıca ölçülecek.

In [3]:
baglanti.close()

baglanti = sqlite3.connect(f"file:{DB_YOLU.resolve().as_posix()}?mode=ro", uri=True)

testler = [
    "SELECT ad, olumlu_oran FROM oyunlar WHERE app_id = 239140",
    "DELETE FROM oyunlar WHERE app_id = -1",
    "UPDATE oyunlar SET ad = ad WHERE app_id = -1",
]

for sorgu in testler:
    print("SORGU:", sorgu)
    print(sql_calistir(sorgu))
    print("-" * 60)

SORGU: SELECT ad, olumlu_oran FROM oyunlar WHERE app_id = 239140
1 satır, 0.00 sn
         ad  olumlu_oran
Dying Light        0.952
------------------------------------------------------------
SORGU: DELETE FROM oyunlar WHERE app_id = -1
HATA: Execution failed on sql 'DELETE FROM oyunlar WHERE app_id = -1': attempt to write a readonly database
------------------------------------------------------------
SORGU: UPDATE oyunlar SET ad = ad WHERE app_id = -1
HATA: Execution failed on sql 'UPDATE oyunlar SET ad = ad WHERE app_id = -1': attempt to write a readonly database
------------------------------------------------------------


### Koruma 1: salt okuma
- Bağlantı `sqlite3.connect("file:<tam yol>?mode=ro", uri=True)` ile açıldı. Yazma kilidi veritabanı seviyesinde: sorgu nasıl yazılırsa yazılsın DELETE/UPDATE/DROP reddedilir.
- Test: SELECT çalıştı; `DELETE` ve `UPDATE` (zararsız, app_id = -1) → "attempt to write a readonly database". Hata metin olarak döndü, agent çökmez.
- Metne bakan "SELECT ile mi başlıyor" kontrolü seçilmedi: kandırılabilir ve `WITH ... SELECT` gibi geçerli sorguları engeller.
- Hız: app_id ile arama 0,00 sn, ad ile arama 2,82 sn (ad indekssiz). Agent oyunu bir kez adıyla bulup sonra app_id ile sorgulamalı.

In [4]:
SATIR_SINIRI = 50

def sql_calistir(sorgu, sinir=SATIR_SINIRI):
    sorgu = sorgu.strip().rstrip(";")
    baslangic = time.perf_counter()
    try:
        imlec = baglanti.execute(sorgu)
        satirlar = imlec.fetchmany(sinir + 1)
        sutunlar = [s[0] for s in imlec.description]
        kesildi = len(satirlar) > sinir
        satirlar = satirlar[:sinir]
        if kesildi:
            toplam = baglanti.execute(f"SELECT COUNT(*) FROM ({sorgu})").fetchone()[0]
    except Exception as hata:
        return f"HATA: {hata}"
    sure = time.perf_counter() - baslangic

    df = pd.DataFrame(satirlar, columns=sutunlar)
    if kesildi:
        ozet = (f"DİKKAT: Sonuç {sinir} satırda kesildi. Sorgunun toplam sonucu {toplam:,} satır, "
                f"sadece ilk {sinir} tanesi gösteriliyor. Sayı lazımsa satırları kendin sayma, "
                f"COUNT(*) kullan. ({sure:.2f} sn)")
    else:
        ozet = f"{len(df)} satır, {sure:.2f} sn"
    return ozet + "\n" + df.to_string(index=False)

testler = [
    "SELECT ad, olumlu_oran, oy_sayisi FROM oyunlar WHERE oy_sayisi >= 1000 ORDER BY olumlu_oran DESC LIMIT 5;",
    "SELECT ad, olumlu_oran, oy_sayisi FROM oyunlar WHERE oy_sayisi >= 100000 ORDER BY olumlu_oran DESC",
    "SELECT review_id, tarih, olumlu FROM yorumlar WHERE app_id = 239140",
    "SELECT ad, puan FROM oyunlar",
]

for sorgu in testler:
    print("SORGU:", sorgu)
    print(sql_calistir(sorgu))
    print("-" * 60)

SORGU: SELECT ad, olumlu_oran, oy_sayisi FROM oyunlar WHERE oy_sayisi >= 1000 ORDER BY olumlu_oran DESC LIMIT 5;
5 satır, 2.91 sn
                                            ad  olumlu_oran  oy_sayisi
Aokana - Four Rhythms Across the Blue - EXTRA2        0.997       1604
                          A Tower Full of Cats        0.995       1574
                     Cats Hidden in Jingle Jam        0.995       9923
                               The WereCleaner        0.995       6584
                         A Castle Full of Cats        0.994       3709
------------------------------------------------------------
SORGU: SELECT ad, olumlu_oran, oy_sayisi FROM oyunlar WHERE oy_sayisi >= 100000 ORDER BY olumlu_oran DESC
DİKKAT: Sonuç 50 satırda kesildi. Sorgunun toplam sonucu 82 satır, sadece ilk 50 tanesi gösteriliyor. Sayı lazımsa satırları kendin sayma, COUNT(*) kullan. (0.07 sn)
                                 ad  olumlu_oran  oy_sayisi
                  Vampire Survivors        0.985   

### Koruma 2: satır sınırı (SATIR_SINIRI = 50)
- Neden: sonuç LLM'e metin olarak gider. Dying Light'ın 179.837 yorumu ≈ 5 milyon karakter, bağlama sığmaz, pahalı; ayrıca hepsini belleğe almak notebook'u kilitler.
- Nasıl: `cursor.fetchmany(sinir + 1)` ile en fazla 51 satır alınır; 51. satır geldiyse sonuç kesilir ve `SELECT COUNT(*) FROM (sorgu)` ile gerçek toplam bulunur. LLM'e "DİKKAT: 50 satırda kesildi, toplam N, sayı için COUNT(*) kullan" yazılır.
- Sınır sadece LLM'e **giden** satırlarda. SQL'in **baktığı** satırlar sınırsız: `COUNT(*)` 179.837 satıra bakar ama 1 satır döndürür.
- Sonuçlar:
  - `LIMIT 5;` → kesilmedi, sondaki `;` silindiği için sorun yok.
  - 100.000+ oylu oyun: 50 gösterildi, **toplam 82**. Uyarı olmasa LLM "50" derdi. COUNT dahil 0,07 sn.
  - Dying Light yorumları: toplam 179.837 (02 ile aynı), 0,01 sn. Gösterilen 50 yorumun **50'si olumlu**: ORDER BY yoksa dosya sırası gelir (Steam "faydalı" sırasına yakın) → kesilmiş sonuç rastgele örnek değil.
  - Hata mesajı kısaldı: "no such column: puan".
- Açıklamaya yazılacak: oran/dağılım SQL'de hesaplanır; "en yeni" için `ORDER BY tarih DESC`.
- Not: ilk tam tabloyu tarayan sorgu 2,91 sn, sonraki 0,07 sn → soğuk başlangıç etkisi.

In [5]:
kullanici_yazimlari = [
    "Dying Light",
    "dying light",
    "elden ring",
    "sims 4",
    "titanfall 2",
    "spiderman",
    "baldurs gate 3",
    "witcher 3",
]

for isim in kullanici_yazimlari:
    print(f"########## Kullanıcı yazdı: '{isim}'")
    print("[= ile]")
    print(sql_calistir(f"SELECT app_id, ad, oy_sayisi FROM oyunlar WHERE ad = '{isim}'"))
    print("[LIKE ile]")
    print(sql_calistir(f"SELECT app_id, ad, oy_sayisi FROM oyunlar WHERE ad LIKE '%{isim}%' ORDER BY oy_sayisi DESC"))
    print()

########## Kullanıcı yazdı: 'Dying Light'
[= ile]
1 satır, 0.03 sn
 app_id          ad  oy_sayisi
 239140 Dying Light     435283
[LIKE ile]
2 satır, 0.03 sn
 app_id                                         ad  oy_sayisi
 239140                                Dying Light     435283
 534380 Dying Light 2 Stay Human: Reloaded Edition     173419

########## Kullanıcı yazdı: 'dying light'
[= ile]
0 satır, 0.03 sn
Empty DataFrame
Columns: [app_id, ad, oy_sayisi]
Index: []
[LIKE ile]
2 satır, 0.03 sn
 app_id                                         ad  oy_sayisi
 239140                                Dying Light     435283
 534380 Dying Light 2 Stay Human: Reloaded Edition     173419

########## Kullanıcı yazdı: 'elden ring'
[= ile]
0 satır, 0.03 sn
Empty DataFrame
Columns: [app_id, ad, oy_sayisi]
Index: []
[LIKE ile]
1 satır, 0.03 sn
 app_id         ad  oy_sayisi
1245620 ELDEN RING     977477

########## Kullanıcı yazdı: 'sims 4'
[= ile]
0 satır, 0.03 sn
Empty DataFrame
Columns: [app_id, ad, o

### Risk 1: oyunu adıyla bulma (= ve LIKE)
- 8 kullanıcı yazımı denendi (7'si veri setinde var, "witcher 3" yok).
- `=` büyük-küçük harfe duyarlı: sadece birebir "Dying Light" bulundu, "dying light" 0.
- `LIKE '%...%'` harf farkını çözdü (dying light, elden ring). Birden fazla sonuç gelebilir (Dying Light + Dying Light 2) → `ORDER BY oy_sayisi DESC`.
- **4 gerçek oyun bulunamadı:** "sims 4" (The Sims™ 4), "titanfall 2" (Titanfall® 2), "spiderman" (Spider-Man), "baldurs gate 3" (Baldur's). Tek karakter (™ ® - ') eşleşmeyi bozuyor → LLM "veri setinde yok" der: sessiz yanlış.
- **Ters hata:** "witcher 3" → "The Witcher 3 REDkit Playtest" (0 oy). Asıl oyun yok; LLM "var" sanabilir.
- Hız: hepsi 0,03 sn. İlk sorgudaki 2,82 sn soğuk başlangıçtı.
- Sonraki adım: (1) "tek ayırt edici kelimeyle ara" kuralı, (2) Python'da benzerlik araması yapan "oyun bul" fonksiyonu; ölçüp seçilecek.

In [6]:
anahtar_kelimeler = {
    "dying light": "dying light",
    "elden ring": "elden",
    "sims 4": "sims",
    "titanfall 2": "titanfall",
    "spiderman": "spider",
    "baldurs gate 3": "baldur",
    "witcher 3": "witcher",
}

for kullanici, kelime in anahtar_kelimeler.items():
    print(f"########## Kullanıcı: '{kullanici}' → aranan: '%{kelime}%'")
    print(sql_calistir(
        f"SELECT app_id, ad, oy_sayisi FROM oyunlar WHERE ad LIKE '%{kelime}%' ORDER BY oy_sayisi DESC",
        sinir=5,
    ))
    print()

########## Kullanıcı: 'dying light' → aranan: '%dying light%'
2 satır, 0.05 sn
 app_id                                         ad  oy_sayisi
 239140                                Dying Light     435283
 534380 Dying Light 2 Stay Human: Reloaded Edition     173419

########## Kullanıcı: 'elden ring' → aranan: '%elden%'
DİKKAT: Sonuç 5 satırda kesildi. Sorgunun toplam sonucu 6 satır, sadece ilk 5 tanesi gösteriliyor. Sayı lazımsa satırları kendin sayma, COUNT(*) kullan. (0.07 sn)
 app_id                           ad  oy_sayisi
1245620                   ELDEN RING     977477
 715020 Elden: Path of the Forgotten         67
2059760              Elden Gunfire 2         58
2768050                Elden Gunfire         10
3143110                  Elden Sword          2

########## Kullanıcı: 'sims 4' → aranan: '%sims%'
1 satır, 0.03 sn
 app_id          ad  oy_sayisi
1222670 The Sims™ 4     153114

########## Kullanıcı: 'titanfall 2' → aranan: '%titanfall%'
1 satır, 0.03 sn
 app_id           ad

### Çözüm 1: "tek ayırt edici kelimeyle ara" kuralı
- Aynı 7 ad, LLM'i taklit ederek tek kelimeyle arandı (`LIKE '%sims%'`, `ORDER BY oy_sayisi DESC`).
- **Kayıp 4 oyunun hepsi bulundu ve en üstte:** sims → The Sims™ 4 (1 sonuç), titanfall → Titanfall® 2 (1), spider → Spider-Man Remastered (44 sonuç içinde 1.), baldur → Baldur's Gate 3 (4 içinde 1.). Oy sıralaması doğru oyunu üste çıkarıyor.
- LIKE kelime parçasını da yakalıyor: "witcher" → Whisker Witchery, Chrome Switcher.
- **Yeni tuzak:** Witcher 3 yok ama en üstte "GWENT: The Witcher Card Game" (28.743 oy) var → LLM GWENT'in oranını Witcher 3 diye verebilir. Playtest kayıtları da ayrı oyun (Baldur's Gate 3 Playtest, 0 oy).
- Sonuç: kural işe yarıyor ama bir ikinci kural şart: "bulunan ad sorulan oyunla aynı mı kontrol et, değilse 'yok' de". Bu LLM'in dikkatine kalıyor (08'de ölçülecek).

In [7]:
import re
from rapidfuzz import process, fuzz

def ad_sadelestir(ad):
    ad = ad.lower()
    ad = re.sub(r"[™®©’'\-]", "", ad)
    ad = re.sub(r"[^\w\s]", " ", ad)
    ad = re.sub(r"\s+", " ", ad).strip()
    return ad

oyun_adlari = pd.read_sql("SELECT app_id, ad, oy_sayisi FROM oyunlar", baglanti)
oyun_adlari["ad_sade"] = oyun_adlari["ad"].fillna("").map(ad_sadelestir)
print(oyun_adlari[oyun_adlari["app_id"].isin([1222670, 1237970, 1817070, 1086940])].to_string(index=False))
print()

def oyun_bul(isim, aday_sayisi=5):
    sonuclar = process.extract(ad_sadelestir(isim), oyun_adlari["ad_sade"], scorer=fuzz.WRatio, limit=20)
    adaylar = pd.DataFrame(
        [(oyun_adlari.loc[sira, "app_id"], oyun_adlari.loc[sira, "ad"], oyun_adlari.loc[sira, "oy_sayisi"], round(puan))
         for _, puan, sira in sonuclar],
        columns=["app_id", "ad", "oy_sayisi", "puan"],
    )
    return adaylar.sort_values(["puan", "oy_sayisi"], ascending=False).head(aday_sayisi)

denemeler = ["dying light", "elden ring", "sims 4", "titanfall 2", "spiderman", "baldurs gate 3", "witcher 3",
             "eldin ring", "sims4", "spider man", "baldurs gat 3"]

for isim in denemeler:
    baslangic = time.perf_counter()
    adaylar = oyun_bul(isim)
    print(f"########## '{isim}' ({time.perf_counter() - baslangic:.2f} sn)")
    print(adaylar.to_string(index=False))
    print()

 app_id                             ad  oy_sayisi                      ad_sade
1086940                Baldur's Gate 3     669941               baldurs gate 3
1222670                    The Sims™ 4     153114                   the sims 4
1237970                   Titanfall® 2     237538                  titanfall 2
1817070 Marvel’s Spider-Man Remastered     101951 marvels spiderman remastered

########## 'dying light' (0.10 sn)
 app_id                                         ad  oy_sayisi  puan
 239140                                Dying Light     435283   100
 534380 Dying Light 2 Stay Human: Reloaded Edition     173419    90
2079960                                      Light          7    90
 925570            The Uncertain: Light At The End       1079    86
1132980        There Is No Light: Enhanced Edition        801    86

########## 'elden ring' (0.10 sn)
 app_id                                         ad  oy_sayisi  puan
1245620                                 ELDEN RING     977

### Çözüm 2: rapidfuzz ile benzerlik araması (oyun_bul)
- Yöntem: adlar sadeleştirildi (küçük harf, ™®©’'- silindi, diğer noktalama → boşluk). Kullanıcının yazdığı da aynı fonksiyondan geçer → "The Sims™ 4" yazılırsa da 100 puan. `process.extract(..., scorer=fuzz.WRatio)` ile 64.948 ad puanlandı, puan + oy sayısına göre ilk 5 aday. Arama 0,10 sn.
- **İyi:** 11 aramanın 9'unda doğru oyun 1. sırada. Yazım hataları bulundu: "eldin ring" → ELDEN RING (90), "baldurs gat 3" → Baldur's Gate 3 (96). LIKE bunları bulamazdı.
- **Sorun 1, boşluk:** "sims4" → The Sims™ 4 3. sırada (72); "spider man" → Spider-Man ilk 5'te yok. Tire silinince "spiderman" oldu, kullanıcı ayrı yazdı.
- **Sorun 2, puan ayırt etmiyor:** kısa adlar kullanıcının yazdığının içinde geçince 90 alıyor ("Ur", "Bal", "GATE" → baldurs gate 3; "FALL" → titanfall 2). Doğru eşleşme de 90, çöp de 90, olmayan Witcher 3 için REDkit Playtest de 90 → puana eşik konamaz.
- Doğru oyunu üste çıkaran puan değil, eşit puanda **oy sayısı** sıralaması.
- Sonraki: boşlukları da silmek; LLM'e 5 aday (ad + oy + puan) gösterip "sorulan oyun değilse 'yok' de" kuralı.

In [8]:
def ad_sadelestir(ad):
    return re.sub(r"[^\w]", "", ad.lower())

oyun_adlari["ad_sade"] = oyun_adlari["ad"].fillna("").map(ad_sadelestir)
print(oyun_adlari[oyun_adlari["app_id"].isin([1222670, 1237970, 1817070, 1086940])].to_string(index=False))
print()

denemeler = ["dying light", "elden ring", "sims 4", "titanfall 2", "spiderman", "baldurs gate 3", "witcher 3",
             "eldin ring", "sims4", "spider man", "baldurs gat 3", "The Sims™ 4"]

for isim in denemeler:
    baslangic = time.perf_counter()
    adaylar = oyun_bul(isim)
    print(f"########## '{isim}' → '{ad_sadelestir(isim)}' ({time.perf_counter() - baslangic:.2f} sn)")
    print(adaylar.to_string(index=False))
    print()

 app_id                             ad  oy_sayisi                    ad_sade
1086940                Baldur's Gate 3     669941               baldursgate3
1222670                    The Sims™ 4     153114                   thesims4
1237970                   Titanfall® 2     237538                 titanfall2
1817070 Marvel’s Spider-Man Remastered     101951 marvelsspidermanremastered

########## 'dying light' → 'dyinglight' (0.08 sn)
 app_id                                         ad  oy_sayisi  puan
 239140                                Dying Light     435283   100
 534380 Dying Light 2 Stay Human: Reloaded Edition     173419    90
2079960                                      Light          7    90
2564240              A Dragon's Tale: Fading Light          4    85
1203680                              Guiding Light        160    82

########## 'elden ring' → 'eldenring' (0.08 sn)
 app_id                                             ad  oy_sayisi  puan
1245620                            

### Çözüm 2, sürüm 2: boşluklar da silindi
- `ad_sadelestir` = `re.sub(r"[^\w]", "", ad.lower())`: harf/rakam dışı her şey silinir. "spider man", "spiderman", "Spider-Man" → "spiderman"; "The Sims™ 4" → "thesims4". Veritabanı tarafındaki `ad_sade` de yeniden hesaplandı (iki taraf aynı fonksiyondan geçmeli).
- **Veri setinde olan 11 aramanın 11'inde doğru oyun 1. sırada.** "sims4" (önce 3.) ve "spider man" (önce ilk 5'te yoktu) düzeldi. "The Sims™ 4" → 100. Yazım hataları: "eldin ring" 89, "baldurs gat 3" 96. ~0,09 sn.
- Puan hâlâ ayırt etmiyor: "Ur", "Bal", "GATE", "P.I.D.", "M/a" 90 alıyor (kullanıcının yazdığının içinde geçiyorlar). Witcher 3 → REDkit Playtest 90.
- Sonuç: fonksiyon doğru oyunu adaylara ve büyük ihtimalle 1. sıraya getiriyor; "oyun var mı" kararını adaylara (ad + oy) bakarak LLM veriyor.

In [9]:
def oyun_bul_araci(isim):
    adaylar = oyun_bul(isim)
    return (
        f"'{isim}' için adı en çok benzeyen 5 oyun (puan = ad benzerliği 0-100, kesin eşleşme anlamına gelmez):\n"
        f"{adaylar.to_string(index=False)}\n"
        "Kural: Sorulan oyunu ad ve oy sayısına bakarak seç. Adaylardan hiçbiri sorulan oyun değilse "
        "(farklı bir oyun, Playtest, araç veya kart oyunu) oyunun veri setinde olmadığını söyle. "
        "Seçtiğin oyunu sonraki SQL sorgularında app_id ile kullan."
    )

for isim in ["witcher 3", "spider man"]:
    print(oyun_bul_araci(isim))
    print("-" * 60)

'witcher 3' için adı en çok benzeyen 5 oyun (puan = ad benzerliği 0-100, kesin eşleşme anlamına gelmez):
 app_id                            ad  oy_sayisi  puan
2889630 The Witcher 3 REDkit Playtest          0    90
1594560               Chrome Switcher          2    84
2669100                       WATCHER          0    80
1284410  GWENT: The Witcher Card Game      28743    79
2376280              Whisker Witchery         64    79
Kural: Sorulan oyunu ad ve oy sayısına bakarak seç. Adaylardan hiçbiri sorulan oyun değilse (farklı bir oyun, Playtest, araç veya kart oyunu) oyunun veri setinde olmadığını söyle. Seçtiğin oyunu sonraki SQL sorgularında app_id ile kullan.
------------------------------------------------------------
'spider man' için adı en çok benzeyen 5 oyun (puan = ad benzerliği 0-100, kesin eşleşme anlamına gelmez):
 app_id                                 ad  oy_sayisi  puan
1817070     Marvel’s Spider-Man Remastered     101951    90
1817190 Marvel’s Spider-Man: Miles Mora

### oyun_bul_araci: LLM'e giden metin
- `oyun_bul` sonucunu metne çevirir: başlık ("puan kesin eşleşme değildir") + 5 aday (app_id, ad, oy_sayisi, puan) + kural ("ad ve oy sayısına bakarak seç; hiçbiri sorulan oyun değilse veri setinde yok de; sonra app_id ile sorgula").
- Kural aracın çıktısında, çünkü LLM onu tam karar anında, adayların altında görüyor.
- "witcher 3": en üstte REDkit Playtest (0 oy, 90), GWENT kart oyunu 4. sırada. 100'e yakın puan yok → beklenen cevap "veri setinde yok".
- "spider man": Remastered (101.951 oy) ve Miles Morales (37.755) ikisi de 90 → gerçek belirsizlik. Beklenen: Remastered'ı seçip Miles Morales'ı anmak veya kullanıcıya sormak.
- 08 test adayları: "Witcher 3 nasıl?" (yok demeli), "Spider-Man'in puanı kaç?" (belirsizliği ele almalı).

In [10]:
oran_karsilastirma = """
SELECT o.ad, o.oy_sayisi, o.oran_kaynagi, ROUND(o.yorum_kapsami, 3) AS yorum_kapsami,
       o.olumlu_oran AS dogru_oran,
       ROUND(AVG(y.olumlu), 3) AS yorumlardan_avg,
       COUNT(*) AS yorum_satiri
FROM oyunlar o
JOIN yorumlar y ON y.app_id = o.app_id
WHERE o.app_id IN (1222670, 1237970, 1172470, 239140, 1086940, 1245620)
GROUP BY o.app_id
ORDER BY ABS(o.olumlu_oran - AVG(y.olumlu)) DESC
"""
print(sql_calistir(oran_karsilastirma))
print()

fark_sayimi = """
SELECT COUNT(*) AS oyun_sayisi,
       SUM(ABS(olumlu_oran - 1.0 * olumlu_csv / (olumlu_csv + olumsuz_csv)) > 0.05) AS fark_5_puandan_fazla,
       SUM(ABS(olumlu_oran - 1.0 * olumlu_csv / (olumlu_csv + olumsuz_csv)) > 0.10) AS fark_10_puandan_fazla
FROM oyunlar
WHERE oy_sayisi >= 100 AND olumlu_csv + olumsuz_csv > 0
"""
print(sql_calistir(fark_sayimi))

6 satır, 0.21 sn
             ad  oy_sayisi oran_kaynagi  yorum_kapsami  dogru_oran  yorumlardan_avg  yorum_satiri
    The Sims™ 4     153114         json          0.060       0.874            0.560          9140
  Apex Legends™     961805         json          0.052       0.677            0.426         50120
    Dying Light     435283         json          0.413       0.952            0.910        179837
   Titanfall® 2     237538         json          0.059       0.953            0.911         14029
     ELDEN RING     977477         json          0.014       0.926            0.894         13770
Baldur's Gate 3     669941         json          0.077       0.968            0.978         51540

1 satır, 2.92 sn
 oyun_sayisi  fark_5_puandan_fazla  fark_10_puandan_fazla
       11513                   113                     41


### Risk 2: oranı yorumlardan AVG ile hesaplamak
- 6 oyunda `oyunlar.olumlu_oran` ile `AVG(yorumlar.olumlu)` karşılaştırıldı (0,21 sn, app_id indeksi).
  - The Sims 4: **%87,4 vs %56,0** (31 puan; "Çok Olumlu" → "Karışık"). Kapsam %6.
  - Apex Legends: **%67,7 vs %42,6** (25 puan). Kapsam %5.
  - Titanfall 2 %95,3 vs 91,1; Dying Light 95,2 vs 91,0; ELDEN RING 92,6 vs 89,4 (kapsam sadece %1,4); Baldur's Gate 3 96,8 vs 97,8.
- Sorun az yorum değil, **hangi yorumların toplandığı** (ELDEN RING %1,4 kapsamla 3 puan fark). 02: Apex zaman etkisi, Sims 4 seçim yanlılığı.
- Bütün veri: oy ≥ 100 ve CSV'li 11.513 oyunun **113'ünde fark > 5 puan, 41'inde > 10 puan** (~%1). Nadir ama etkilenenler popüler oyunlar. (Toplu hesap oyunlar.olumlu_csv/olumsuz_csv ile, 31,7 M satır okunmadan.)
- yorumlar.olumlu yasaklanamaz: dil/tarih/süre alt grupları sadece oradan hesaplanır.
- **Kural:** genel oran → `oyunlar.olumlu_oran`. Alt grup → yorumlar'dan AVG, cevapta "elimizdeki N yorumdan" de; `yorum_kapsami` düşükse örneklem olduğunu söyle.

In [11]:
kelimeler = ["crash", "freeze", "bug", "çök", "Çök", "donuyor"]

for k in kelimeler:
    print(f"########## Dying Light, metin LIKE '%{k}%'")
    print(sql_calistir(f"SELECT COUNT(*) AS yorum FROM yorumlar WHERE app_id = 239140 AND metin LIKE '%{k}%'"))
    print()

print("########## 'crash' geçen ilk 8 yorum")
print(sql_calistir("""
    SELECT olumlu, substr(metin, 1, 200) AS metin_ilk_200
    FROM yorumlar
    WHERE app_id = 239140 AND metin LIKE '%crash%'
    LIMIT 8
"""))
print()

print("########## Bütün yorumlarda 'crash' (birkaç dakika sürebilir)")
print(sql_calistir("SELECT COUNT(*) AS yorum FROM yorumlar WHERE metin LIKE '%crash%'"))

########## Dying Light, metin LIKE '%crash%'
1 satır, 0.08 sn
 yorum
   472

########## Dying Light, metin LIKE '%freeze%'
1 satır, 0.04 sn
 yorum
    72

########## Dying Light, metin LIKE '%bug%'
1 satır, 0.04 sn
 yorum
  3259

########## Dying Light, metin LIKE '%çök%'
1 satır, 0.07 sn
 yorum
    39

########## Dying Light, metin LIKE '%Çök%'
1 satır, 0.06 sn
 yorum
     0

########## Dying Light, metin LIKE '%donuyor%'
1 satır, 0.04 sn
 yorum
     3

########## 'crash' geçen ilk 8 yorum
8 satır, 0.00 sn
 olumlu                                                                                                                                                                                            metin_ilk_200
      1 🔸I GIVE THIS GAME A 8/10Dying light without a doubt is the best zombie game out there. I have been playing this since the release and till date i play this game.Back then i use to play this on my PS4 
      1 Update on my Old Review (2020-10-04):I would still recommend 

### Risk 3: yorum içeriğini `metin LIKE` ile aramak
- Dying Light'ta "çökme" konusu, kelimeye göre: crash 472, freeze 72, bug 3.259, çök 39, Çök 0, donuyor 3. Tek doğru sayı yok.
- "bug" kelime parçası da yakalıyor ("bugün", "debug"). LIKE Türkçe büyük/küçük harfi eşlemiyor (çök ≠ Çök). 27 dilin çoğu aranmadı bile.
- "crash" geçen 8 yorumun **7'si olumlu**; biri "60 FPS No Crashing". Kelimenin geçmesi ≠ şikâyet (olumsuzlama, geçerken bahsetme).
- Bütün 31,7 M yorumda `LIKE '%crash%'`: **22,4 sn**, 331.185. Birkaç sorguda dakikayı geçer → **sorgu zaman aşımı** koruması adayı.
- Sonuç: içerik/konu soruları RAG aracının işi (anlam araması, çok dilli). SQL'de LIKE ile sayım yanıltıcı.

In [12]:
deneme_sorgulari = {
    "50 oyun, ad + oran": "SELECT ad, olumlu_oran FROM oyunlar WHERE oy_sayisi >= 100000",
    "1 oyun, SELECT *": "SELECT * FROM oyunlar WHERE app_id = 239140",
    "50 oyun, açıklamalı": "SELECT ad, kisa_aciklama, oyun_hakkinda FROM oyunlar WHERE oy_sayisi >= 100000",
    "50 yorum, SELECT *": "SELECT * FROM yorumlar WHERE app_id = 239140",
}

for etiket, sorgu in deneme_sorgulari.items():
    sonuc = sql_calistir(sorgu)
    karakter = len(sonuc)
    bosluk_orani = sonuc.count(" ") / karakter
    print(f"{etiket:22} → {karakter:>9,} karakter, ~{karakter // 4:>8,} token, boşluk %{bosluk_orani * 100:.0f}")

50 oyun, ad + oran     →     3,021 karakter, ~     755 token, boşluk %62
1 oyun, SELECT *       →     3,984 karakter, ~     996 token, boşluk %53
50 oyun, açıklamalı    →   291,783 karakter, ~  72,945 token, boşluk %69
50 yorum, SELECT *     →   184,127 karakter, ~  46,031 token, boşluk %88


### Risk 4: metin sütunlarını çekmek (LLM'e giden yazı)
| Sorgu | Karakter | ~Token | Boşluk |
|---|---|---|---|
| 50 oyun, ad + oran | 3.021 | 755 | %62 |
| 1 oyun, SELECT * | 3.984 | 996 | %53 |
| 50 oyun, açıklamalı | 291.783 | 72.945 | %69 |
| 50 yorum, SELECT * | 184.127 | 46.031 | %88 |
- Satır sınırı satır uzunluğunu sınırlamıyor: açıklamalı 50 oyun ~73 bin, 50 yorum ~46 bin token → yerel 7B/14B bağlamına (8–32 bin) sığmaz.
- **Sürpriz:** çıktının %53–88'i boşluk. `to_string` her sütunu en uzun değere göre hizalıyor, kısa değerleri boşlukla dolduruyor. LLM'e hizalama gerekmez → CSV biçimi.
- Token ≈ karakter / 4 (kaba; Türkçe/Çince için gerçek sayı daha yüksek).

In [13]:
METIN_SUTUNLARI = {("yorumlar", "metin"), ("oyunlar", "kisa_aciklama"), ("oyunlar", "oyun_hakkinda")}
METIN_MODU = "gizle"
KESME_UZUNLUGU = 200

def metin_bekcisi(islem, tablo, sutun, veritabani, kaynak):
    if islem == sqlite3.SQLITE_READ and (tablo, sutun) in METIN_SUTUNLARI:
        return sqlite3.SQLITE_DENY
    return sqlite3.SQLITE_OK

def hucre_kisalt(deger):
    if isinstance(deger, str) and len(deger) > KESME_UZUNLUGU:
        return deger[:KESME_UZUNLUGU] + f"...[kesildi, toplam {len(deger)} karakter]"
    return deger

def sql_calistir(sorgu, sinir=SATIR_SINIRI):
    baglanti.set_authorizer(metin_bekcisi if METIN_MODU == "gizle" else None)
    sorgu = sorgu.strip().rstrip(";")
    baslangic = time.perf_counter()
    try:
        imlec = baglanti.execute(sorgu)
        satirlar = imlec.fetchmany(sinir + 1)
        sutunlar = [s[0] for s in imlec.description]
        kesildi = len(satirlar) > sinir
        satirlar = satirlar[:sinir]
        if kesildi:
            toplam = baglanti.execute(f"SELECT COUNT(*) FROM ({sorgu})").fetchone()[0]
    except Exception as hata:
        mesaj = f"HATA: {hata}"
        if "prohibited" in str(hata):
            mesaj += (" (Yorum metni ve oyun açıklamaları SQL aracında kapalı. İçerik için RAG aracını kullan; "
                      "sayı, oran ve filtre için bu sütunlar olmadan sorgu yaz.)")
        return mesaj
    sure = time.perf_counter() - baslangic

    df = pd.DataFrame(satirlar, columns=sutunlar)
    if METIN_MODU == "kes":
        df = df.map(hucre_kisalt)
    if kesildi:
        ozet = (f"DİKKAT: Sonuç {sinir} satırda kesildi. Sorgunun toplam sonucu {toplam:,} satır, "
                f"sadece ilk {sinir} tanesi gösteriliyor. Sayı lazımsa satırları kendin sayma, "
                f"COUNT(*) kullan. ({sure:.2f} sn)")
    else:
        ozet = f"{len(df)} satır, {sure:.2f} sn"
    return ozet + "\n" + df.to_csv(index=False)

deneme_sorgulari["Dying Light, metin LIKE"] = "SELECT COUNT(*) FROM yorumlar WHERE app_id = 239140 AND metin LIKE '%crash%'"

for mod in ["gizle", "kes"]:
    METIN_MODU = mod
    print(f"=============== METIN_MODU = '{mod}'")
    for etiket, sorgu in deneme_sorgulari.items():
        sonuc = sql_calistir(sorgu)
        if sonuc.startswith("HATA"):
            print(f"{etiket:24} → {sonuc}")
        else:
            karakter = len(sonuc)
            print(f"{etiket:24} → {karakter:>9,} karakter, ~{karakter // 4:>8,} token, boşluk %{sonuc.count(' ') / karakter * 100:.0f}")
    print()

METIN_MODU = "kes"
print("=============== 'kes' modunda 50 yorumun ilk 1.000 karakteri")
print(sql_calistir("SELECT * FROM yorumlar WHERE app_id = 239140")[:1000])
METIN_MODU = "gizle"

=============== METIN_MODU = 'gizle'
50 oyun, ad + oran       →     1,308 karakter, ~     327 token, boşluk %8
1 oyun, SELECT *         → HATA: access to oyunlar.kisa_aciklama is prohibited (Yorum metni ve oyun açıklamaları SQL aracında kapalı. İçerik için RAG aracını kullan; sayı, oran ve filtre için bu sütunlar olmadan sorgu yaz.)
50 oyun, açıklamalı      → HATA: access to oyunlar.kisa_aciklama is prohibited (Yorum metni ve oyun açıklamaları SQL aracında kapalı. İçerik için RAG aracını kullan; sayı, oran ve filtre için bu sütunlar olmadan sorgu yaz.)
50 yorum, SELECT *       → HATA: access to yorumlar.metin is prohibited (Yorum metni ve oyun açıklamaları SQL aracında kapalı. İçerik için RAG aracını kullan; sayı, oran ve filtre için bu sütunlar olmadan sorgu yaz.)
Dying Light, metin LIKE  → HATA: access to yorumlar.metin is prohibited (Yorum metni ve oyun açıklamaları SQL aracında kapalı. İçerik için RAG aracını kullan; sayı, oran ve filtre için bu sütunlar olmadan sorgu yaz.)

======

In [14]:
ZAMAN_ASIMI_SN = 10

def sql_calistir_zamanli(sorgu, sinir=SATIR_SINIRI):
    bitis = time.perf_counter() + ZAMAN_ASIMI_SN
    baglanti.set_progress_handler(lambda: 1 if time.perf_counter() > bitis else 0, 10_000)
    try:
        sonuc = sql_calistir(sorgu, sinir)
    finally:
        baglanti.set_progress_handler(None, 0)
    if sonuc.startswith("HATA: interrupted"):
        sonuc = (f"HATA: Sorgu {ZAMAN_ASIMI_SN} saniyeyi geçtiği için durduruldu. Daha dar bir sorgu yaz: "
                 "app_id veya indeksli sütunlarla (dil, tarih) filtrele, bütün yorumlar tablosunu tarayan sorgulardan kaçın.")
    return sonuc

testler = [
    ("gizle", "Normal sorgu", "SELECT COUNT(*) AS turkce FROM yorumlar WHERE dil = 'tr'"),
    ("gizle", "Bütün yorumların ortalama oynama süresi", "SELECT AVG(oynama_saat) FROM yorumlar"),
    ("kes",   "Bütün yorumlarda LIKE", "SELECT COUNT(*) FROM yorumlar WHERE metin LIKE '%crash%'"),
]

for mod, etiket, sorgu in testler:
    METIN_MODU = mod
    baslangic = time.perf_counter()
    sonuc = sql_calistir_zamanli(sorgu)
    print(f"########## [{mod}] {etiket} ({time.perf_counter() - baslangic:.1f} sn)")
    print(sonuc)
    print()

METIN_MODU = "gizle"

########## [gizle] Normal sorgu (0.0 sn)
1 satır, 0.03 sn
turkce
619054


########## [gizle] Bütün yorumların ortalama oynama süresi (8.2 sn)
1 satır, 8.15 sn
AVG(oynama_saat)
105.09320392047466


########## [kes] Bütün yorumlarda LIKE (6.7 sn)
1 satır, 6.73 sn
COUNT(*)
331185




In [15]:
ZAMAN_ASIMI_SN = 2

for mod, etiket, sorgu in testler[1:]:
    METIN_MODU = mod
    baslangic = time.perf_counter()
    sonuc = sql_calistir_zamanli(sorgu)
    print(f"########## [{mod}] {etiket} ({time.perf_counter() - baslangic:.1f} sn)")
    print(sonuc)
    print()

ZAMAN_ASIMI_SN = 10
METIN_MODU = "gizle"

########## [gizle] Bütün yorumların ortalama oynama süresi (2.0 sn)
HATA: Sorgu 2 saniyeyi geçtiği için durduruldu. Daha dar bir sorgu yaz: app_id veya indeksli sütunlarla (dil, tarih) filtrele, bütün yorumlar tablosunu tarayan sorgulardan kaçın.

########## [kes] Bütün yorumlarda LIKE (2.0 sn)
HATA: Sorgu 2 saniyeyi geçtiği için durduruldu. Daha dar bir sorgu yaz: app_id veya indeksli sütunlarla (dil, tarih) filtrele, bütün yorumlar tablosunu tarayan sorgulardan kaçın.



### Koruma 5: zaman aşımı (ZAMAN_ASIMI_SN = 10)
- `sql_calistir_zamanli`: `baglanti.set_progress_handler(fonksiyon, 10_000)` ile SQLite her 10.000 adımda saati kontrol eder, süre dolduysa sorguyu kendisi durdurur ("interrupted"). `finally` ile bildirim her durumda kaldırılır. LLM'e: "durduruldu, app_id / indeksli sütunlarla daralt".
- 10 sn ile: tr sayımı 0,03 sn; tüm yorumlarda AVG(oynama_saat) 8,2 sn (105,1 saat); tüm yorumlarda LIKE '%crash%' 6,7 sn. Hiçbiri durmadı.
- Aynı LIKE ilk seferde 22,4 sn, şimdi 6,7 sn: **önbellek** (veri RAM'de). Süre makinenin durumuna göre 3 kat değişiyor; Cloud Run'da yavaş taraf beklenir.
- 2 sn ile test: iki ağır sorgu da **2,0 sn'de** durdu, açıklayıcı mesaj döndü. Mekanizma doğrulandı.
- Metinsiz sorgular da ağır olabilir (tüm yorumlarda AVG). Oyun başına süre için `oyunlar.yorumcu_medyan_saat` kullanılmalı (açıklamaya).
- Sınır değeri 08'de gerçek sorularla gözden geçirilecek.

In [16]:
deger_sorgulari = {
    "oyun_kategori.kategori": "SELECT kategori AS deger, COUNT(*) AS oyun FROM oyun_kategori GROUP BY kategori ORDER BY oyun DESC",
    "oyun_tur.tur": "SELECT tur AS deger, COUNT(*) AS oyun FROM oyun_tur GROUP BY tur ORDER BY oyun DESC",
    "oyun_dil.dil (ilk 30)": "SELECT dil AS deger, COUNT(*) AS oyun, SUM(ses_var) AS sesli FROM oyun_dil GROUP BY dil ORDER BY oyun DESC LIMIT 30",
    "yorumlar.dil": "SELECT dil AS deger, COUNT(*) AS yorum FROM yorumlar GROUP BY dil ORDER BY yorum DESC",
    "oyunlar.oran_kaynagi": "SELECT oran_kaynagi AS deger, COUNT(*) AS oyun FROM oyunlar GROUP BY oran_kaynagi ORDER BY oyun DESC",
    "oyun_etiket.etiket (ilk 40)": "SELECT etiket AS deger, COUNT(*) AS oyun FROM oyun_etiket GROUP BY etiket ORDER BY oyun DESC LIMIT 40",
}

for etiket, sorgu in deger_sorgulari.items():
    baslangic = time.perf_counter()
    tablo = pd.read_sql(sorgu, baglanti)
    print(f"########## {etiket}: {len(tablo)} değer ({time.perf_counter() - baslangic:.2f} sn)")
    print(tablo.to_string(index=False))
    print()

########## oyun_kategori.kategori: 39 değer (0.05 sn)
                     deger  oyun
             Single-player 57338
            Family Sharing 50065
        Steam Achievements 27006
               Steam Cloud 14005
   Full controller support 13047
              Multi-player 10204
Partial Controller Support  6923
                       PvP  6676
                     Co-op  5876
                Online PvP  4801
        Steam Leaderboards  3922
              Online Co-op  3895
       Shared/Split Screen  3765
      Remote Play Together  3531
   Shared/Split Screen PvP  2480
                   VR Only  2443
       Steam Trading Cards  2418
 Shared/Split Screen Co-op  2255
Tracked Controller Support  2253
                     Stats  1952
          In-App Purchases  1647
Cross-Platform Multiplayer  1464
            Steam Workshop  1075
     Includes level editor   997
         Remote Play on TV   782
                       MMO   774
                   LAN PvP   710
                 LAN C

### Sütun değerlerinin yazılışı (araç açıklaması için)
- **Kategori (39), iç içe:** Multi-player (10.204) ⊃ PvP (6.676) / Co-op (5.876) ⊃ Online Co-op (3.895), Shared/Split Screen Co-op (2.255), LAN Co-op (657); PvP için aynısı. "İnternetten co-op" = `'Online Co-op'`. Yazılış: `'Single-player'`, `'Multi-player'`.
- **Tür (30):** geliştirici seçiyor, kaba. Indie 42.275. "Early Access", "Free To Play" de tür. Korku türü yok.
- **Etiket (oyuncular):** `'Singleplayer'`, `'Multiplayer'` (kategoriden farklı yazılış!), `'Horror'` 5.751.
- **İki dil sütunu:** `oyun_dil.dil` tam ad (`'Turkish'` 5.864 oyun, 2.035 sesli; `'Spanish - Spain'`, `'Portuguese - Brazil'`). `yorumlar.dil` kod (`'tr'` 619.054; `'bilinmiyor'` 3.371.327 = %11).
- **oran_kaynagi:** json 34.968, csv 15.144, boş 14.836 (oy yok). 02'deki sayılardan fark = 738 yazılım.
- **Açık konu:** oran_kaynagi GROUP BY 31,9 sn (sabah benzer tarama 0,07 sn). Büyük yorum taramaları sonrası oyunlar tablosu bellekten atılmış olabilir; açıklamalar satırların içinde olduğu için tablo ağır. Soğuk durumda 10 sn zaman aşımı normal sorguları da durdurabilir → 08 ve deploy'da ölçülecek.

In [17]:
SQL_ARAC_ACIKLAMASI = """Steam oyun veritabanında (SQLite) salt okunur SQL sorgusu çalıştırır. Veri 2020-2024 arası Steam oyunlarını ve 2024 sonuna kadarki oyuncu yorumlarını kapsar.
Sonuç CSV olarak döner. En fazla 50 satır gösterilir; kesildiyse toplam satır sayısı yazılır. 10 saniyeyi geçen sorgu durdurulur.

NE ZAMAN KULLANILIR: sayı, oran, sıralama, filtreleme, sayım.
- Yorumların İÇERİĞİ (oyuncular ne diyor, şikâyetler, çökme var mı) için bu aracı değil RAG aracını kullan.
- Oyunu adıyla SQL'de arama. Önce oyun_bul aracıyla app_id'yi bul, sonra WHERE app_id = ... kullan.

TABLOLAR
oyunlar (64.948 oyun, birincil anahtar app_id)
- app_id, ad
- olumlu_oran: oyunun GENEL olumlu oy oranı, 0-1 arası (0.874 = %87,4). Oyunun puanı için TEK doğru kaynak. NULL = oy yok.
- oy_sayisi: toplam oy, popülerlik ölçüsü. oran_kaynagi: 'json', 'csv' veya NULL.
- yorum_sayisi: veritabanındaki yorum satırı sayısı. yorum_var: 1/0. yorum_kapsami: yorum_sayisi / oy_sayisi (0.06 = oyların %6'sının yorumu elimizde).
- olumlu_json, olumsuz_json, olumlu_csv, olumsuz_csv: ham sayılar. Oran için KULLANMA, olumlu_oran kullan.
- fiyat: verinin toplandığı günkü fiyat (indirimli olabilir). normal_fiyat: indirimsiz fiyat, fiyat sorularında bunu kullan. indirim_yuzde. Ücretsiz oyun: fiyat = 0.
- steam_cikis_tarihi ('YYYY-AA-GG'): Steam sayfasındaki tarih, oyunun ilk çıkışı OLMAYABİLİR. Oyunun ne zaman çıktığı için ilk_yorum_tarihi kullan. son_yorum_tarihi.
- sahip_alt, sahip_ust: tahmini sahip sayısı aralığı. peak_ccu: aynı anda en yüksek oyuncu sayısı. istatistik_eksik = 1 ise bu sütunlar ve oy bilgisi NULL.
- yorumcu_medyan_saat: yorum yazanların oynama süresi medyanı (saat). Oynama süresi sorularında bunu kullan. ortalama_sure_saat ve medyan_sure_saat sadece ~3.500 oyunda dolu. yorumcu_sure_sifir_orani.
- yas_siniri: sadece 422 oyunda dolu.
- tr_yorum_sayisi: Türkçe yorum sayısı. chroma_yorum_sayisi: RAG'da aranabilen yorum sayısı.
- Bayraklar (1/0): yetiskin (yetişkin içerik), iskelet (sayfası boş kayıt), dil_spam ve ses_spam (geliştirici neredeyse bütün dilleri işaretlemiş; dil bilgisine güvenme).
- kisa_aciklama, oyun_hakkinda: bu araçta KAPALI.

oyun_tur (app_id, tur): geliştiricinin seçtiği kaba tür. 'Action', 'Adventure', 'Casual', 'Indie', 'RPG', 'Simulation', 'Strategy', 'Sports', 'Racing', 'Massively Multiplayer', 'Early Access', 'Free To Play'. Korku türü yoktur.
oyun_etiket (app_id, etiket, oy, sira): oyuncuların verdiği etiketler, sira 1 = en çok oy alan. Tür benzeri sorularda (korku, roguelike vb.) etiket kullan, ör. 'Horror'. Yazılış: 'Singleplayer', 'Multiplayer' (tiresiz).
oyun_kategori (app_id, kategori): Steam özellikleri, iç içe: 'Multi-player' ⊃ 'PvP' ve 'Co-op' ⊃ 'Online PvP', 'Online Co-op', 'Shared/Split Screen PvP', 'Shared/Split Screen Co-op', 'LAN PvP', 'LAN Co-op'. Diğerleri: 'Single-player', 'Shared/Split Screen', 'Cross-Platform Multiplayer', 'MMO', 'VR Only', 'VR Supported', 'Full controller support', 'Steam Achievements'. İnternetten birlikte oynama = 'Online Co-op'.
oyun_dil (app_id, dil, ses_var): geliştiricinin beyan ettiği diller, İngilizce tam ad: 'English', 'Turkish', 'German', 'Simplified Chinese', 'Spanish - Spain', 'Portuguese - Brazil'. ses_var = 1 seslendirme var. Cevapta "geliştirici belirtmiş" de.

yorumlar (31,7 milyon yorum, birincil anahtar review_id)
- app_id, tarih ('YYYY-AA-GG'), olumlu (1/0), oynama_saat (yorum yazılırken), faydali_oy, erken_erisim (1/0), uzunluk (karakter).
- dil: tespit edilen dil KODU: 'en', 'zh', 'ru', 'es', 'pt', 'de', 'ko', 'tr', 'fr', 'ja' ... veya 'bilinmiyor'. (oyun_dil'deki 'Turkish' ile karıştırma.) dil_ham, dil_guven.
- resim, kopya (çöp ve kopya yorum, 1/0), chroma_uygun.
- metin: bu araçta KAPALI.
- İndeksli sütunlar: app_id, dil, tarih.

KURALLAR
1. Oyunun genel puanı = oyunlar.olumlu_oran. yorumlar.olumlu'dan AVG alma (yorumu eksik oyunlarda çok yanlış çıkar: The Sims 4 %87 yerine %56).
2. Alt grup oranı (dile, tarihe, oynama süresine göre) için yorumlar'dan AVG(olumlu) kullanılabilir. Cevapta "elimizdeki N yorumdan" de; yorum_kapsami 0.2'den küçükse bunun örneklem olduğunu belirt.
3. "En iyi / en yüksek puanlı" sıralamalarında az oylu oyunları ele (en az oy_sayisi >= 100).
4. Sayı lazımsa COUNT(*) kullan, kesilmiş listeyi sayma. ORDER BY yazmazsan satırların sırası anlamsızdır.
5. Tarihi aralıkla filtrele: tarih >= '2024-01-01' AND tarih < '2025-01-01'. strftime kullanma (yavaş).
6. yorumlar sorgularını app_id, dil veya tarih ile daralt. Bütün yorumları tarayan sorgu zaman aşımına uğrar.
7. Kullanıcı açıkça istemedikçe yetiskin = 1 veya iskelet = 1 olan oyunları önerme.
8. Bir değerin yazılışından emin değilsen önce kontrol et, ör. SELECT DISTINCT etiket FROM oyun_etiket WHERE etiket LIKE '%rogue%'.
9. Sonuç 0 satırsa önce yazılışı kontrol et, sonra "veri yok" de.
10. Oranları kullanıcıya yüzde olarak söyle (0.952 → %95).
"""

print(f"{len(SQL_ARAC_ACIKLAMASI):,} karakter, ~{len(SQL_ARAC_ACIKLAMASI) // 4:,} token")

4,844 karakter, ~1,211 token


### SQL aracının açıklaması (SQL_ARAC_ACIKLAMASI)
- LLM'in veritabanı hakkında bildiği her şey bu metin. Bölümler: aracın sınırları (50 satır, CSV, 10 sn) → ne zaman kullanılır (içerik → RAG, oyun adı → oyun_bul) → tablolar ve sütunların anlamı/tuzakları → 10 kural.
- Her kural bir ölçümden geliyor: olumlu_oran tek kaynak (Risk 2, Sims 4 %87 vs %56), kesilmiş listeyi sayma (satır sınırı), tarih aralığı (03), yorumlar sorgusunu daralt (zaman aşımı), yazılışı kontrol et / 0 satırda "veri yok" deme (Risk 1), yetişkin/iskelet önerme (02), kategori hiyerarşisi ve 'Turkish'/'tr' (değer listesi).
- Boyut: 4.844 karakter, ~1.211 token (Türkçe → gerçekte ~1.500–2.000). Her soruda gönderilir; küçük modelde (8k bağlam) ~%25.
- "gizle" moduna göre yazıldı (metin sütunları KAPALI).

In [18]:
import json

SQL_ARACI = {
    "type": "function",
    "function": {
        "name": "sql_calistir",
        "description": SQL_ARAC_ACIKLAMASI,
        "parameters": {
            "type": "object",
            "properties": {
                "sorgu": {"type": "string", "description": "Çalıştırılacak tek bir SQLite SELECT sorgusu."},
            },
            "required": ["sorgu"],
        },
    },
}

OYUN_BUL_ARACI = {
    "type": "function",
    "function": {
        "name": "oyun_bul",
        "description": ("Oyun adını benzerlikle arar ve adı en çok benzeyen 5 oyunu (app_id, ad, oy_sayisi, puan) döndürür. "
                        "Kullanıcı bir oyunun adını verdiğinde SQL'den ÖNCE bunu kullan, doğru oyunu seçip app_id'sini SQL'de kullan. "
                        "Adaylardan hiçbiri sorulan oyun değilse oyun veri setinde yoktur."),
        "parameters": {
            "type": "object",
            "properties": {
                "isim": {"type": "string", "description": "Kullanıcının yazdığı oyun adı, olduğu gibi (ör. 'sims 4')."},
            },
            "required": ["isim"],
        },
    },
}

ARAC_TANIMLARI = [SQL_ARACI, OYUN_BUL_ARACI]

ARAC_FONKSIYONLARI = {
    "sql_calistir": lambda arg: sql_calistir_zamanli(arg["sorgu"]),
    "oyun_bul": lambda arg: oyun_bul_araci(arg["isim"]),
}

def arac_cagir(ad, argumanlar_json):
    if ad not in ARAC_FONKSIYONLARI:
        return f"HATA: '{ad}' diye bir araç yok. Kullanılabilir araçlar: {', '.join(ARAC_FONKSIYONLARI)}."
    try:
        argumanlar = json.loads(argumanlar_json)
    except json.JSONDecodeError:
        return "HATA: Argümanlar geçerli JSON değil. Örnek: {\"sorgu\": \"SELECT ...\"}"
    try:
        return ARAC_FONKSIYONLARI[ad](argumanlar)
    except KeyError as eksik:
        return f"HATA: Eksik argüman: {eksik}. Aracın parametre adlarını kontrol et."

print(f"Bütün araç tanımları: ~{len(json.dumps(ARAC_TANIMLARI, ensure_ascii=False)) // 4:,} token\n")

llm_istekleri = [
    ("oyun_bul", '{"isim": "sims 4"}'),
    ("sql_calistir", '{"sorgu": "SELECT ad, olumlu_oran, oy_sayisi, yorum_kapsami FROM oyunlar WHERE app_id = 1222670"}'),
    ("sql_calistir", '{"query": "SELECT 1"}'),
    ("web_ara", '{"q": "sims 4"}'),
    ("sql_calistir", '{"sorgu": "SELECT ad FROM oyunlar'),
]

for ad, argumanlar in llm_istekleri:
    print(f"########## LLM isteği: {ad}({argumanlar})")
    print(arac_cagir(ad, argumanlar))
    print()

Bütün araç tanımları: ~1,417 token

########## LLM isteği: oyun_bul({"isim": "sims 4"})
'sims 4' için adı en çok benzeyen 5 oyun (puan = ad benzerliği 0-100, kesin eşleşme anlamına gelmez):
 app_id                      ad  oy_sayisi  puan
1222670             The Sims™ 4     153114    90
2209440               sim4D ATC        110    80
3118260 SIM - Simple Idle Miner          8    80
3103230              Sim Solver          0    80
1313490                  Slimes         37    73
Kural: Sorulan oyunu ad ve oy sayısına bakarak seç. Adaylardan hiçbiri sorulan oyun değilse (farklı bir oyun, Playtest, araç veya kart oyunu) oyunun veri setinde olmadığını söyle. Seçtiğin oyunu sonraki SQL sorgularında app_id ile kullan.

########## LLM isteği: sql_calistir({"sorgu": "SELECT ad, olumlu_oran, oy_sayisi, yorum_kapsami FROM oyunlar WHERE app_id = 1222670"})
1 satır, 0.01 sn
ad,olumlu_oran,oy_sayisi,yorum_kapsami
The Sims™ 4,0.874,153114,0.06


########## LLM isteği: sql_calistir({"query": "SELECT

### Tool tanımı ve arac_cagir
- OpenAI uyumlu biçim (Ollama, DeepSeek, Gemini'nin OpenAI uyumlu adresi kabul ediyor): `{"type": "function", "function": {"name", "description", "parameters": JSON Schema}}`.
- `sql_calistir(sorgu)`: açıklama = SQL_ARAC_ACIKLAMASI. `oyun_bul(isim)`: kısa açıklama, kurallar çıktının içinde.
- `ARAC_FONKSIYONLARI`: araç adı → Python fonksiyonu (zaman aşımlı SQL, oyun_bul_araci). `arac_cagir(ad, argumanlar_json)` hatalı isteklerde çökmez, düzeltme ipucu döner.
- Test (taklit LLM istekleri):
  - oyun_bul("sims 4") → The Sims™ 4 1. (153.114 oy); SQL app_id 1222670 → olumlu_oran 0.874, yorum_kapsami 0.06, 0,01 sn.
  - `query` (yanlış parametre) → "Eksik argüman: 'sorgu'"; `web_ara` → araç listesi; yarım JSON → doğru biçim örneği.
- Araç tanımlarının toplamı ~1.417 token.

In [19]:
baglanti.close()
print("Bağlantı kapandı.")
print(f"Ayarlar: SATIR_SINIRI={SATIR_SINIRI}, ZAMAN_ASIMI_SN={ZAMAN_ASIMI_SN}, METIN_MODU='{METIN_MODU}', KESME_UZUNLUGU={KESME_UZUNLUGU}")
print("Araçlar:", [t["function"]["name"] for t in ARAC_TANIMLARI])

Bağlantı kapandı.
Ayarlar: SATIR_SINIRI=50, ZAMAN_ASIMI_SN=10, METIN_MODU='gizle', KESME_UZUNLUGU=200
Araçlar: ['sql_calistir', 'oyun_bul']
